# BRGI Datathon 2026 — district BRGI quantile forecasts

**Approach**
1. **What BRGI is.** BRGI = 35·rank(credit yoy) + 20·rank(deposit yoy) + 30·rank(night-light yoy) + 15·rank(resilience); this was verified exactly on the `score_*_l1` columns. So the features are mostly *forecasts of those future cross-sectional ranks*:
   * credit and deposit: the already-realised part of next year's growth, plus a projection at the recent growth rate, rebuilt from `bank_quarterly.csv`
   * night lights: a seasonal-level projection (the trailing 4-quarter level plus the district's seasonal profile), compared with the base quarter
   * resilience: a proxy, rank(mean − 0.5·sd of the last 4 credit qoq growths), which correlates about 0.83 with the score
2. **Data availability is respected** (from `release_calendar.csv`). At origin O, bank data and BRGI labels are used only up to O−1, and night lights only up to the 2nd month of O. No external data is used.
3. **Models.** LightGBM quantile regression (p10/p50/p90), with horizon as a feature. There is a *full* model for districts with history and a *no-bank* model (night lights, static data, neighbours) for the ~15% unseen districts. Two variants are blended 50/50: a direct target, and a residual over the last BRGI.
   * `train.csv` has no rows for origins 2024Q1–Q3, so those rows are **rebuilt from the raw files** (bank, night lights and labels; every column checked against existing origins). The weather, fire and news columns can't be rebuilt, so they are dropped. In backtests they added nothing. Several seeds are averaged, the p10/p90 distance from p50 is scaled by ×0.85 (2025 turned out calmer than past years), and the quantiles are sorted and clipped to [0, 100].
4. **Validation.** Rolling-origin backtests at 2021Q4, 2022Q4 and 2023Q4, with 15% of districts held out entirely to simulate unseen ones, scored on horizon-weighted pinball loss.

No AutoML: all cleaning, feature engineering and training code is written by hand. Fully deterministic (fixed seeds, `deterministic=True`).

In [1]:
import os, warnings, json
import numpy as np, pandas as pd, lightgbm as lgb
warnings.filterwarnings('ignore')
# data path: Kaggle or local
for cand in ['/kaggle/input/brgi-datathon-2026/', '/mnt/user-data/uploads/Buisnesss/', './']:
    if os.path.exists(os.path.join(cand, 'train.csv')):
        DATA = cand; break
print('DATA =', DATA, '| lightgbm', lgb.__version__)
RUN_BACKTEST = False       # set True to reproduce the validation scores (slower)
N_THREADS = os.cpu_count()

DATA = /mnt/user-data/uploads/Buisnesss/ | lightgbm 4.7.0


## 1. Helpers: quarter arithmetic and the competition metric

In [2]:
"""Shared helpers: data loading, quarter math, metric."""

QS = [0.1, 0.5, 0.9]
HW = {1: 1.0, 2: 1.0, 3: 1.25, 4: 1.25}
SEED = 42


def qidx(s):
    """'2024Q4' -> integer quarter index."""
    s = pd.Series(s).astype(str)
    return (s.str[:4].astype(int) * 4 + s.str[-1].astype(int) - 1).values


def qstr(i):
    i = np.asarray(i)
    return np.char.add(np.char.add((i // 4).astype(str), 'Q'), (i % 4 + 1).astype(str))


def pinball(y, p, q):
    d = y - p
    return np.maximum(q * d, (q - 1) * d)


def score(y, P10, P50, P90, h):
    """Horizon-weighted mean pinball, averaged over the 3 quantiles."""
    w = np.vectorize(HW.get)(h)
    out = []
    for q, p in zip(QS, (P10, P50, P90)):
        out.append(np.sum(w * pinball(y, p, q)) / np.sum(w))
    return float(np.mean(out))


def load():
    tr = pd.read_csv(DATA + 'train.csv')
    te = pd.read_csv(DATA + 'test.csv')
    return tr, te

## 2. Feature engineering (point-in-time correct)

In [3]:
"""Feature engineering for the BRGI datathon.

Every feature for a row (district d, origin O, horizon h, target T=O+h) uses only
information available at origin O, per release_calendar.csv:
  * BRGI labels / component scores: up to quarter O-1 (1-quarter lag)
  * bank_quarterly: up to quarter O-1
  * nightlights_monthly: up to the 2nd month of quarter O (1-month lag)
BRGI = 35*credit_yoy_rank + 20*deposit_yoy_rank + 30*ntl_yoy_rank + 15*resilience_rank,
so many features are cross-sectional ranks / projections of those components.
"""

SCORE_COLS = ['score_credit_yoy_l1', 'score_deposit_yoy_l1', 'score_ntl_yoy_l1', 'score_resilience_l1']


def _matrix(df, row, col, val, rows, ncol, col0):
    M = np.full((len(rows), ncol), np.nan)
    ri = pd.Index(rows).get_indexer(df[row])
    ci = df[col].values - col0
    ok = (ri >= 0) & (ci >= 0) & (ci < ncol)
    M[ri[ok], ci[ok]] = df[val].values[ok]
    return M


def _rank_cols(M):
    """Percentile rank of each column (cross-section of districts), NaN-aware."""
    return pd.DataFrame(M).rank(pct=True).values


def build(tr, te, bank=None):
    tr = tr.copy(); te = te.copy()
    tr['_is_test'] = 0; te['_is_test'] = 1
    df = pd.concat([tr, te], ignore_index=True)
    df['O'] = qidx(df.origin_quarter)
    df['T'] = df.O + df.horizon
    dists = sorted(set(df.district_id) | set(pd.read_csv(DATA + 'district_static.csv').district_id))
    dix = pd.Index(dists).get_indexer(df.district_id)
    Q0 = qidx(['2004Q1'])[0]; NQ = qidx(['2025Q4'])[0] - Q0 + 1
    O = df.O.values; T = df['T'].values; h = df.horizon.values

    def gather(M, q, valid):
        q = np.asarray(q); out = np.full(len(df), np.nan)
        c = q - Q0
        ok = valid & (c >= 0) & (c < M.shape[1])
        out[ok] = M[dix[ok], c[ok]]
        return out

    F = {}
    # ---------- BRGI label history (labels only up to O-1) ----------
    lab = tr[['district_id', 'target_quarter', 'brgi']].drop_duplicates(['district_id', 'target_quarter'])
    lab['q'] = qidx(lab.target_quarter)
    B = _matrix(lab, 'district_id', 'q', 'brgi', dists, NQ, Q0)
    for k in range(3, 9):
        F[f'brgi_l{k}'] = gather(B, O - k, np.ones(len(df), bool))
    F['brgi_T4'] = gather(B, T - 4, T - 4 <= O - 1)
    F['brgi_T8'] = gather(B, T - 8, T - 8 <= O - 1)
    # expanding stats up to O-1
    cs = np.nancumsum(np.nan_to_num(B), axis=1); cn = np.cumsum(~np.isnan(B), axis=1)
    cs2 = np.nancumsum(np.nan_to_num(B) ** 2, axis=1)
    s = gather(cs, O - 1, np.ones(len(df), bool)); n = gather(cn.astype(float), O - 1, np.ones(len(df), bool))
    s2 = gather(cs2, O - 1, np.ones(len(df), bool))
    with np.errstate(invalid='ignore', divide='ignore'):
        F['brgi_exp_mean'] = np.where(n > 0, s / n, np.nan)
        F['brgi_exp_std'] = np.where(n > 1, np.sqrt(np.maximum(s2 / n - (s / n) ** 2, 0)), np.nan)
    F['brgi_hist_n'] = np.nan_to_num(n)
    lags = np.column_stack([df.brgi_l1.values, df.brgi_l2.values] + [F[f'brgi_l{k}'] for k in range(3, 9)])
    with np.errstate(all='ignore'):
        import warnings; warnings.simplefilter('ignore')
        F['brgi_mean8'] = np.nanmean(lags, axis=1)
    F['brgi_slope4'] = df.brgi_l1.values - F['brgi_l4']

    # ---------- component-score history (score_*_l1 at origin O == score at O-1) ----------
    for c in SCORE_COLS:
        sc = df[['district_id', 'O', c]].dropna().drop_duplicates(['district_id', 'O'])
        sc['q'] = sc.O - 1
        S = _matrix(sc, 'district_id', 'q', c, dists, NQ, Q0)
        base = c.replace('score_', '').replace('_l1', '')
        F[f's_{base}_T4'] = gather(S, T - 4, T - 4 <= O - 1)
        F[f's_{base}_l4'] = gather(S, O - 4, np.ones(len(df), bool))
        F[f's_{base}_l2'] = gather(S, O - 2, np.ones(len(df), bool))

    # ---------- bank reconstruction (up to O-1) ----------
    bq = pd.read_csv(DATA + 'bank_quarterly.csv') if bank is None else bank.copy()
    bq['q'] = qidx(bq.quarter)
    bq['lc'] = np.log(bq.credit); bq['ld'] = np.log(bq.deposit); bq['lo'] = np.log(bq.offices)
    LC = _matrix(bq, 'district_id', 'q', 'lc', dists, NQ, Q0)
    LD = _matrix(bq, 'district_id', 'q', 'ld', dists, NQ, Q0)
    HD = _matrix(bq, 'district_id', 'q', 'hdfc_merger_window', dists, NQ, Q0)
    for name, L in (('c', LC), ('d', LD)):
        Y = np.full_like(L, np.nan); Y[:, 4:] = L[:, 4:] - L[:, :-4]
        R = _rank_cols(Y)
        for k in (1, 2, 3, 4):
            F[f'{name}r_l{k}'] = gather(R, O - k, np.ones(len(df), bool))
        F[f'{name}r_T4'] = gather(R, T - 4, T - 4 <= O - 1)
        F[f'tgt_{name}r'] = gather(R, T, np.ones(len(df), bool))  # analysis only, never a feature
        F[f'{name}y_T4'] = gather(Y, T - 4, T - 4 <= O - 1)
        last = gather(L, O - 1, np.ones(len(df), bool))
        l5 = gather(L, O - 5, np.ones(len(df), bool))
        l3 = gather(L, O - 3, np.ones(len(df), bool))
        g4 = (last - l5) / 4.0                      # avg qoq over last year
        g2 = (last - l3) / 2.0                      # avg qoq over last 2 quarters
        base = gather(L, T - 4, T - 4 <= O - 1)     # level at T-4 if known
        base = np.where(T - 4 <= O - 1, base, last + (T - 4 - (O - 1)) * g4)
        F[f'{name}_partial'] = np.where(T - 4 <= O - 2, last - base, np.nan)  # realised part of yoy(T)
        F[f'{name}_est_yoy'] = (last + (T - (O - 1)) * g4) - base
        F[f'{name}_est_yoy2'] = (last + (T - (O - 1)) * (0.5 * g4 + 0.5 * g2)) - base
        # seasonal: known part + median of the same-calendar-span growth in past years
        rem = np.column_stack([gather(L, T - 4 * k, T - 4 * k <= O - 1) - gather(L, O - 1 - 4 * k, np.ones(len(df), bool))
                               for k in (1, 2, 3)])
        pastyoy = np.column_stack([gather(L, T - 4 * k, T - 4 * k <= O - 1) - gather(L, T - 4 * k - 4, T - 4 * k <= O - 1)
                                   for k in (2, 3, 4)])
        with np.errstate(all='ignore'):
            import warnings; warnings.simplefilter('ignore')
            R = np.nanmedian(rem, axis=1); PY = np.nanmedian(pastyoy, axis=1)
        known = np.where(T - 4 <= O - 1, last - gather(L, T - 4, T - 4 <= O - 1), np.nan)
        F[f'{name}_seas_est_yoy'] = np.where(T - 4 <= O - 1, known + R, 0.5 * PY + 0.5 * 4 * g4)
    # resilience proxy: rank of (mean - 0.5*sd) of the last 4 credit qoq growths (corr ~0.83 with score)
    QQ = np.full_like(LC, np.nan); QQ[:, 1:] = LC[:, 1:] - LC[:, :-1]
    W = np.stack([np.roll(QQ, k, axis=1) for k in range(4)], axis=2); W[:, :3, :] = np.nan
    with np.errstate(all='ignore'):
        RES = _rank_cols(np.nanmean(W, axis=2) - 0.5 * np.nanstd(W, axis=2, ddof=1))
    for k in (1, 2, 4):
        F[f'resp_l{k}'] = gather(RES, O - k, np.ones(len(df), bool))
    F['resp_T4'] = gather(RES, T - 4, T - 4 <= O - 1)
    F['tgt_res'] = gather(RES, T, np.ones(len(df), bool))  # analysis only
    # known part of the window that defines resilience at T (qoq at T-3..O-1)
    kq = np.column_stack([np.where(T - j <= O - 1, gather(QQ, T - j, T - j <= O - 1), np.nan) for j in range(4)])
    with np.errstate(all='ignore'):
        F['res_part_mean'] = np.nanmean(kq, axis=1)
        F['res_part_min'] = np.nanmin(kq, axis=1)
    F['hdfc_T4'] = gather(HD, T - 4, T - 4 <= O - 1)
    F['hdfc_l1'] = gather(HD, O - 1, np.ones(len(df), bool))

    # ---------- night lights (known to 2nd month of quarter O) ----------
    nl = pd.read_csv(DATA + 'nightlights_monthly.csv')
    nl['m'] = nl.month.str[:4].astype(int) * 12 + nl.month.str[5:7].astype(int) - 1
    M0 = nl.m.min(); NM = (2025 * 12 + 12) - M0
    NLm = _matrix(nl, 'district_id', 'm', 'ntl_sum', dists, NM, M0)
    with np.errstate(divide='ignore', invalid='ignore'):
        LNm = np.log(np.maximum(NLm, 1e-3))
    # quarterly log-mean (complete quarters)
    qm = np.arange(NQ) + Q0  # quarter index -> first month
    NQL = np.full((len(dists), NQ), np.nan)
    for j, q in enumerate(qm):
        fm = (q // 4) * 12 + (q % 4) * 3 - M0
        if fm >= 0 and fm + 3 <= NM:
            blk = NLm[:, fm:fm + 3]
            NQL[:, j] = np.log(np.maximum(blk.mean(axis=1), 1e-3))
    NY = np.full_like(NQL, np.nan); NY[:, 4:] = NQL[:, 4:] - NQL[:, :-4]
    NR = _rank_cols(NY)
    for k in (1, 2, 3, 4):
        F[f'nr_l{k}'] = gather(NR, O - k, np.ones(len(df), bool))
    F['nr_T4'] = gather(NR, T - 4, T - 4 <= O - 1)
    F['tgt_nr'] = gather(NR, T, np.ones(len(df), bool))  # analysis only, never a feature
    F['ny_T4'] = gather(NY, T - 4, T - 4 <= O - 1)
    # partial current quarter: first 2 months of quarter O vs same 2 months a year earlier
    fmO = (O // 4) * 12 + (O % 4) * 3 - M0
    def mgather(M, mi):
        out = np.full(len(df), np.nan); ok = (mi >= 0) & (mi < M.shape[1])
        out[ok] = M[dix[ok], mi[ok]]; return out
    cur2 = np.log(np.maximum((mgather(NLm, fmO) + mgather(NLm, fmO + 1)) / 2, 1e-3))
    prv2 = np.log(np.maximum((mgather(NLm, fmO - 12) + mgather(NLm, fmO - 11)) / 2, 1e-3))
    F['ntl_cur_yoy2m'] = cur2 - prv2
    # level of last 3 known months vs T-4 quarter level (seasonality-naive)
    last3 = np.log(np.maximum((mgather(NLm, fmO - 1) + mgather(NLm, fmO) + mgather(NLm, fmO + 1)) / 3, 1e-3))
    ntlT4 = gather(NQL, T - 4, T - 4 <= O)  # quarter T-4 complete only if <= O-1 (O partially)
    ntlT4 = np.where(T - 4 <= O - 1, ntlT4, np.nan)
    F['ntl_base_gap'] = ntlT4 - gather(NQL, T - 8, T - 8 <= O - 1)  # yoy at T-4 (base effect)
    # seasonal-adjusted projection: yoy at T ~ recent yoy momentum
    F['ntl_est_yoy'] = np.where(np.isnan(F['ntl_cur_yoy2m']), gather(NY, O - 1, np.ones(len(df), bool)), F['ntl_cur_yoy2m'])
    # seasonal-level projection: est log NTL(T) = trailing-4q level + district seasonal factor of T's quarter
    nd = len(dists); Y4 = NQL.reshape(nd, NQ // 4, 4)                       # (district, year, qtr)
    DEV = Y4 - np.nanmean(Y4, axis=2, keepdims=True)                         # seasonal deviation in a year
    yO = (O - Q0) // 4                                                       # index of origin's year
    qT = (T - Q0) % 4
    seas = np.full(len(df), 0.0); cnt = np.zeros(len(df))
    for k in (1, 2, 3):                                                      # last 3 complete years (< origin year)
        yy = yO - k; ok = yy >= 0
        v = np.full(len(df), np.nan); v[ok] = DEV[dix[ok], yy[ok], qT[ok]]
        good = ~np.isnan(v); seas[good] += v[good]; cnt[good] += 1
    seas = np.where(cnt > 0, seas / np.maximum(cnt, 1), np.nan)
    lvl4 = np.nanmean(np.column_stack([gather(NQL, O - k, np.ones(len(df), bool)) for k in (1, 2, 3, 4)]), axis=1)
    est_T = lvl4 + seas
    qT4 = (T - 4 - Q0) % 4
    seas4 = np.full(len(df), 0.0); cnt4 = np.zeros(len(df))
    for k in (1, 2, 3):
        yy = yO - k; ok = yy >= 0
        v = np.full(len(df), np.nan); v[ok] = DEV[dix[ok], yy[ok], qT4[ok]]
        good = ~np.isnan(v); seas4[good] += v[good]; cnt4[good] += 1
    seas4 = np.where(cnt4 > 0, seas4 / np.maximum(cnt4, 1), np.nan)
    base_T4 = np.where(T - 4 <= O - 1, gather(NQL, T - 4, T - 4 <= O - 1),
                       np.where(np.isnan(cur2), lvl4 + seas4, cur2))   # h=4: quarter O from its 2 known months
    F['ntl_seas_est_yoy'] = est_T - base_T4
    F['ntl_base_dev'] = base_T4 - (lvl4 + seas4)          # how abnormal the comparison base is
    F['ntl_lvl_vs_last12'] = last3 - lvl4

    # ---------- cross-sectional ranks within (origin, horizon) ----------
    tmp = pd.DataFrame({'O': O, 'h': h})
    for c in ['c_partial', 'd_partial', 'c_est_yoy', 'd_est_yoy', 'c_est_yoy2', 'd_est_yoy2', 'ntl_cur_yoy2m', 'ntl_est_yoy', 'ntl_base_gap', 'ntl_seas_est_yoy', 'ntl_base_dev', 'c_seas_est_yoy', 'd_seas_est_yoy', 'res_part_mean']:
        tmp[c] = F[c]
        F[c + '_rk'] = tmp.groupby(['O', 'h'])[c].rank(pct=True).values
    res = df.score_resilience_l1.values
    F['nowcast'] = 100 * (0.35 * F['c_est_yoy_rk'] + 0.20 * F['d_est_yoy_rk'] + 0.30 * F['ntl_est_yoy_rk'] + 0.15 * res)
    F['nowcast3'] = 100 * (0.35 * F['c_est_yoy_rk'] + 0.20 * F['d_est_yoy_rk'] + 0.30 * F['ntl_seas_est_yoy_rk'] + 0.15 * res)
    F['nowcast2'] = 100 * (0.35 * F['c_seas_est_yoy_rk'] + 0.20 * F['d_seas_est_yoy_rk'] + 0.30 * F['ntl_seas_est_yoy_rk'] + 0.15 * res)
    F['nowcast_nores'] = 100 * (0.35 * F['c_est_yoy_rk'] + 0.20 * F['d_est_yoy_rk'] + 0.30 * F['ntl_est_yoy_rk']) / 0.85

    feat = pd.DataFrame(F, index=df.index)
    df = pd.concat([df, feat], axis=1)

    # ---------- static / lineage / graph ----------
    st = pd.read_csv(DATA + 'district_static.csv')
    extra = [c for c in st.columns if c not in df.columns]
    df = df.merge(st[['district_id'] + extra], on='district_id', how='left')
    li = pd.read_csv(DATA + 'district_lineage.csv')
    df = df.merge(li, on='district_id', how='left')
    g = pd.read_csv(DATA + 'district_graph.csv')
    deg = g.groupby('district_id').agg(nbr_n=('neighbor_id', 'size'), nbr_dist_mean=('distance_km', 'mean'),
                                       nbr_same_state=('same_state', 'mean')).reset_index()
    df = df.merge(deg, on='district_id', how='left')

    # ---------- neighbour aggregates (inverse-distance weighted, same origin/horizon) ----------
    nb_cols = ['brgi_l1', 'brgi_mean4', 'brgi_exp_mean', 'brgi_T4', 'nowcast', 'nowcast2', 'nowcast3', 'c_est_yoy_rk', 'd_est_yoy_rk', 'ntl_seas_est_yoy_rk', 'c_seas_est_yoy_rk',
               'ntl_est_yoy_rk', 'ntl_cur_yoy2m_rk', 'score_resilience_l1', 'score_credit_yoy_l1', 's_resilience_T4']
    key = df[['district_id', 'O', 'horizon'] + nb_cols].drop_duplicates(['district_id', 'O', 'horizon'])
    gg = g.copy(); gg['w'] = 1.0 / np.maximum(gg.distance_km, 10.0)
    m = gg.merge(key.rename(columns={'district_id': 'neighbor_id'}), on='neighbor_id')
    agg = {}
    for c in nb_cols:
        v = m[c]; ok = v.notna()
        m[c + '_wv'] = np.where(ok, v * m.w, 0.0); m[c + '_ww'] = np.where(ok, m.w, 0.0)
    grp = m.groupby(['district_id', 'O', 'horizon'])
    out = grp[[c + '_wv' for c in nb_cols] + [c + '_ww' for c in nb_cols]].sum()
    nb = pd.DataFrame(index=out.index)
    for c in nb_cols:
        nb['nbw_' + c] = out[c + '_wv'] / out[c + '_ww'].replace(0, np.nan)
    nb = nb.reset_index()
    df = df.merge(nb, on=['district_id', 'O', 'horizon'], how='left')
    # state-level mean of seen districts' BRGI at origin (leave-one-out not needed: own value excluded below)
    sg = df.drop_duplicates(['district_id', 'O']).groupby(['state', 'O']).brgi_l1.agg(['sum', 'count']).reset_index()
    df = df.merge(sg, on=['state', 'O'], how='left')
    own = df.brgi_l1.fillna(0); ownc = df.brgi_l1.notna().astype(int)
    df['state_mean_brgi_l1'] = (df['sum'] - own) / (df['count'] - ownc).replace(0, np.nan)
    df = df.drop(columns=['sum', 'count'])
    df['region_c'] = df.region.astype('category').cat.codes
    df['state_c'] = df.state.astype('category').cat.codes
    return df

## 3. Quantile models

In [4]:
"""LightGBM quantile models: a 'full' model for seen districts and a 'nobank'
model (no own banking/BRGI history) for unseen districts."""

DROP_ALWAYS = {'id', 'district_id', 'state', 'region', 'origin_quarter', 'target_quarter', 'brgi',
               '_is_test', 'O', 'T'}
OWN_BANK_PREFIX = ('brgi_', 'score_', 'cr_', 'dr_', 'cy_', 'dy_', 'c_', 'd_', 'hdfc_', 'nowcast', 'resp_', 'res_part')
OWN_BANK_COLS = {'log_credit', 'log_deposit', 'log_offices', 'cd_ratio', 'credit_per_capita', 'offices_per_100k',
                 'credit_yoy_l1', 'credit_yoy_l2', 'deposit_yoy_l1', 'deposit_yoy_l2', 'offices_yoy_l1',
                 'credit_qoq_l1', 'credit_qoq_std8', 'credit_yoy_mean4', 'bank_gap_in_window', 'rrb_adjusted_l1'}


def feature_sets(df):
    num = [c for c in df.columns if c not in DROP_ALWAYS and df[c].dtype != object and not c.startswith(('s_', 'tgt_'))
           and c != 'nbw_s_resilience_T4']
    full = num
    nobank = [c for c in num if c not in OWN_BANK_COLS and not c.startswith(OWN_BANK_PREFIX)]
    return full, nobank


PARAMS = dict(learning_rate=0.03, num_leaves=31, min_data_in_leaf=80, feature_fraction=0.7,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED,
              deterministic=True, force_row_wise=True, num_threads=N_THREADS)


def anchor_values(d, anchor):
    """Baseline the trees model residuals around (None -> 0)."""
    if anchor is None:
        return np.zeros(len(d))
    if anchor == 'l1':
        return d.brgi_l1.fillna(d.nbw_brgi_l1).fillna(50).values
    if anchor == 'mix':
        return (0.5 * d.brgi_l1.fillna(d.nbw_brgi_l1) + 0.5 * d.nowcast3.fillna(d.nbw_nowcast3)).fillna(50).values
    if anchor == 'nbr':
        return d.nbw_brgi_l1.fillna(50).values
    if anchor == 'nbrmix':
        return (0.5 * d.nbw_brgi_l1.fillna(50) + 25).values
    raise ValueError(anchor)


def fit_predict(trn, tst, feats, rounds, params=None, seeds=(SEED,), anchor=None):
    """Return dict q -> predictions on tst."""
    params = dict(params or {})
    out = {}
    tail = params.pop('tail', {})
    hl = params.pop('w_halflife', None); wq4 = params.pop('w_q4', 1.0); wh = params.pop('w_h', False)
    w = np.ones(len(trn))
    if hl:
        w *= 0.5 ** ((trn.O.max() - trn.O.values) / hl)
    if wq4 != 1.0:
        w *= np.where(trn.O.values % 4 == 3, wq4, 1.0)
    if wh:
        w *= np.where(trn.horizon.values >= 3, 1.25, 1.0)
    a_tr, a_te = anchor_values(trn, anchor), anchor_values(tst, anchor)
    for q in QS:
        p = np.zeros(len(tst))
        for s in seeds:
            prm = dict(PARAMS, **(params or {}), objective='quantile', alpha=q, seed=s)
            if q != 0.5:
                prm.update(tail)
            ds = lgb.Dataset(trn[feats], trn.brgi.values - a_tr, weight=w)
            m = lgb.train(prm, ds, num_boost_round=rounds[q] if isinstance(rounds, dict) else rounds)
            p += (m.predict(tst[feats]) + a_te) / len(seeds)
        out[q] = p
    return out


def finalize(p10, p50, p90):
    P = np.sort(np.c_[p10, p50, p90], axis=1)
    P = np.clip(P, 0.0, 100.0)
    return P[:, 0], P[:, 1], P[:, 2]

## 4. Rebuild the missing recent origins (2024Q1-Q3)

In [5]:
"""Rebuild train-style rows for forecast origins that are missing from train.csv
(2024Q1-2024Q3 for the final model). Every column is computed point-in-time from the raw
files exactly as the organisers' columns are (verified against existing origins).
Labels come from train.csv targets; component scores at O-1 are reconstructed ranks."""

ENV = ['temp_anomaly_c', 'precip_anomaly_pct', 'heat_days_40c', 'heat_days_40c_4q', 'precip_anomaly_pct_4q',
       'max_dry_spell_days', 'log_fire_count', 'fire_count_yoy', 'log_news_events', 'news_share_yoy', 'news_tone',
       'ev_protest_share', 'ev_violence_share']


def _bank_panel():
    bq = pd.read_csv(DATA + 'bank_quarterly.csv')
    bq['q'] = qidx(bq.quarter)
    bq = bq.sort_values(['district_id', 'q'])
    g = bq.groupby('district_id')
    bq['lc'] = np.log(bq.credit); bq['ld'] = np.log(bq.deposit); bq['lo'] = np.log(bq.offices)
    bq['cyoy'] = bq.lc - g.lc.shift(4); bq['dyoy'] = bq.ld - g.ld.shift(4); bq['oyoy'] = bq.lo - g.lo.shift(4)
    bq['cqoq'] = bq.lc - g.lc.shift(1)
    g = bq.groupby('district_id')
    bq['cqoq_std8'] = g.cqoq.transform(lambda s: s.rolling(8, min_periods=8).std())
    bq['cyoy_mean4'] = g.cyoy.transform(lambda s: s.rolling(4, min_periods=4).mean())
    bq['gap_win'] = g.bank_gap.transform(lambda s: s.rolling(8, min_periods=1).max())
    return bq


def _ntl_quarterly():
    nl = pd.read_csv(DATA + 'nightlights_monthly.csv')
    nl['q'] = qidx(nl.month.str[:4] + 'Q' + ((nl.month.str[5:7].astype(int) - 1) // 3 + 1).astype(str))
    agg = nl.groupby(['district_id', 'q']).agg(ntl=('ntl_sum', 'sum'), n=('ntl_sum', 'count'),
                                               gf=('ntl_quality_prop_2_gapfilled', 'mean')).reset_index()
    agg = agg[agg.n == 3]
    return nl, agg


def build_rows(origins, tr, te):
    """Return a DataFrame with train.csv columns for the given origin strings (seen districts only)."""
    bq = _bank_panel()
    nl, nq = _ntl_quarterly()
    st = pd.read_csv(DATA + 'district_static.csv')
    g = pd.read_csv(DATA + 'district_graph.csv')
    lab = tr[['district_id', 'target_quarter', 'brgi']].drop_duplicates(['district_id', 'target_quarter'])
    lab['q'] = qidx(lab.target_quarter)
    L = lab.set_index(['district_id', 'q']).brgi
    # population: per district per origin-year, taken from the nearest existing rows
    allrows = pd.concat([tr, te])
    allrows['year'] = allrows.origin_quarter.str[:4].astype(int)
    pop = allrows.drop_duplicates(['district_id', 'year'])[['district_id', 'year', 'log_population', 'pop_growth_3y',
                                                            'log_pop_density']]
    seen = sorted(set(tr.district_id))
    rows = []
    for o in origins:
        O = qidx([o])[0]
        base = pd.DataFrame({'district_id': seen})
        b1 = bq[bq.q == O - 1].set_index('district_id'); b2 = bq[bq.q == O - 2].set_index('district_id')
        d = base.set_index('district_id')
        d['log_credit'] = b1.lc; d['log_deposit'] = b1.ld; d['log_offices'] = b1.lo
        d['cd_ratio'] = b1.credit / b1.deposit
        d['credit_yoy_l1'] = b1.cyoy; d['credit_yoy_l2'] = b2.cyoy
        d['deposit_yoy_l1'] = b1.dyoy; d['deposit_yoy_l2'] = b2.dyoy
        d['offices_yoy_l1'] = b1.oyoy; d['credit_qoq_l1'] = b1.cqoq
        d['credit_qoq_std8'] = b1.cqoq_std8; d['credit_yoy_mean4'] = b1.cyoy_mean4
        d['bank_gap_in_window'] = b1.gap_win; d['rrb_adjusted_l1'] = b1.rrb_adjusted
        d['_credit'] = b1.credit; d['_offices'] = b1.offices
        # BRGI lags
        d['brgi_l1'] = [L.get((k, O - 1), np.nan) for k in d.index]
        d['brgi_l2'] = [L.get((k, O - 2), np.nan) for k in d.index]
        d['brgi_mean4'] = np.nanmean(np.column_stack([[L.get((k, O - j), np.nan) for k in d.index] for j in (1, 2, 3, 4)]), axis=1)
        # component scores at O-1 (cross-sectional ranks among districts with data)
        d['score_credit_yoy_l1'] = d.credit_yoy_l1.rank(pct=True)
        d['score_deposit_yoy_l1'] = d.deposit_yoy_l1.rank(pct=True)
        n1 = nq[nq.q == O - 1].set_index('district_id'); n2 = nq[nq.q == O - 2].set_index('district_id')
        n5 = nq[nq.q == O - 5].set_index('district_id'); n6 = nq[nq.q == O - 6].set_index('district_id')
        d['log_ntl_l1'] = np.log(n1.ntl.reindex(d.index) / 3)
        d['ntl_yoy_l1'] = d.log_ntl_l1 - np.log(n5.ntl.reindex(d.index))
        d['ntl_yoy_l2'] = np.log(n2.ntl.reindex(d.index)) - np.log(n6.ntl.reindex(d.index))
        d['ntl_qoq_l1'] = d.log_ntl_l1 - np.log(n2.ntl.reindex(d.index))
        d['ntl_gapfilled_share_l1'] = n1.gf.reindex(d.index)
        # partial: months of quarter O known (first 2) vs same months a year earlier
        nl['mi'] = nl.month.str[:4].astype(int) * 12 + nl.month.str[5:7].astype(int) - 1
        fm = (O // 4) * 12 + (O % 4) * 3
        cur = nl[nl.mi.isin([fm, fm + 1])].groupby('district_id').ntl_sum.sum()
        prv = nl[nl.mi.isin([fm - 12, fm - 11])].groupby('district_id').ntl_sum.sum()
        d['ntl_yoy_partial'] = np.log(cur.reindex(d.index)) - np.log(prv.reindex(d.index))
        # ntl component: rank among ALL districts with ntl (incl. unseen)
        nr_all = (np.log(n1.ntl) - np.log(n5.ntl.reindex(n1.index))).rank(pct=True)
        d['score_ntl_yoy_l1'] = nr_all.reindex(d.index)
        # resilience = residual of the exact BRGI identity
        res = (d.brgi_l1 - 35 * d.score_credit_yoy_l1 - 20 * d.score_deposit_yoy_l1 - 30 * d.score_ntl_yoy_l1) / 15
        d['score_resilience_l1'] = res.clip(0, 1)
        d = d.reset_index()
        rows.append((O, d))
    out = []
    for O, d in rows:
        for h in (1, 2, 3, 4):
            T = O + h
            y = [L.get((k, T), np.nan) for k in d.district_id]
            r = d.copy(); r['brgi'] = y; r['horizon'] = h
            r['origin_quarter'] = qstr([O])[0]; r['target_quarter'] = qstr([T])[0]
            r['target_qtr_of_year'] = T % 4 + 1
            out.append(r[r.brgi.notna()])
    out = pd.concat(out, ignore_index=True)
    out['year'] = out.origin_quarter.str[:4].astype(int)
    out = out.merge(pop, on=['district_id', 'year'], how='left')
    # fall back to the closest year if missing
    out = out.merge(st, on='district_id', how='left', suffixes=('', '_st'))
    out['credit_per_capita'] = out._credit * 1e7 / np.exp(out.log_population)
    out['offices_per_100k'] = out._offices / (np.exp(out.log_population) / 1e5)
    # neighbour means of the l1 signals (simple mean over neighbours present)
    key = out.drop_duplicates(['district_id', 'origin_quarter'])[['district_id', 'origin_quarter', 'brgi_l1', 'credit_yoy_l1', 'ntl_yoy_l1']]
    m = g.merge(key.rename(columns={'district_id': 'neighbor_id'}), on='neighbor_id')
    nb = m.groupby(['district_id', 'origin_quarter'])[['brgi_l1', 'credit_yoy_l1', 'ntl_yoy_l1']].mean()
    nb.columns = ['nbr_mean_brgi_l1', 'nbr_mean_credit_yoy_l1', 'nbr_mean_ntl_yoy_l1']
    out = out.merge(nb.reset_index(), on=['district_id', 'origin_quarter'], how='left')
    for c in ENV:
        out[c] = np.nan
    out['id'] = out.district_id + '_' + out.origin_quarter + '_h' + out.horizon.astype(str)
    out = out[[c for c in tr.columns]]
    return out

## 5. Build features

In [6]:
tr, te = load()
aug = build_rows(['2024Q1', '2024Q2', '2024Q3'], tr, te)
print('rebuilt rows:', len(aug), aug.origin_quarter.value_counts().to_dict())
trm = pd.concat([tr, aug], ignore_index=True); trm[ENV] = np.nan
te = te.copy(); te[ENV] = np.nan
df = build(trm, te)
full, nobank = feature_sets(df)
full = [c for c in full if c not in ENV]; nobank = [c for c in nobank if c not in ENV]
print(df.shape, '| full features:', len(full), '| no-bank features:', len(nobank))

rebuilt rows: 3171 {'2024Q1': 1585, '2024Q2': 1057, '2024Q3': 529}


(68636, 188) | full features: 148 | no-bank features: 71


## 6. (Optional) rolling-origin backtest

In [7]:
CONFIGS = [dict(num_leaves=15, r_seen=600, r_un=400),
           dict(num_leaves=15, r_seen=600, r_un=400, anchor='l1', anchor_un='nbr')]
WIDEN = 0.85   # scale of the p10/p90 distance from p50 (best on the leaderboard)

def run_cfgs(trn, tst_seen, tst_un, seeds):
    acc_s = np.zeros((len(tst_seen), 3)); acc_u = np.zeros((len(tst_un), 3))
    for cfg in CONFIGS:
        cfg = dict(cfg); r_s, r_u = cfg.pop('r_seen'), cfg.pop('r_un')
        an, anu = cfg.pop('anchor', None), cfg.pop('anchor_un', None)
        ps = fit_predict(trn, tst_seen, full, r_s, params=cfg, seeds=seeds, anchor=an)
        pu = fit_predict(trn, tst_un, nobank, r_u, params=cfg, seeds=seeds, anchor=anu)
        acc_s += np.column_stack([ps[q] for q in QS]) / len(CONFIGS)
        acc_u += np.column_stack([pu[q] for q in QS]) / len(CONFIGS)
    return acc_s, acc_u

def widen_finalize(P):
    p10 = P[:, 1] - WIDEN * (P[:, 1] - P[:, 0]); p90 = P[:, 1] + WIDEN * (P[:, 2] - P[:, 1])
    return np.column_stack(finalize(p10, P[:, 1], p90))

if RUN_BACKTEST:
    lab = df[df._is_test == 0]; rng = np.random.default_rng(SEED)
    for V in ['2021Q4', '2022Q4', '2023Q4']:
        v = qidx([V])[0]
        dists = lab[lab.O == v].district_id.unique()
        pseudo = set(rng.choice(dists, int(0.15 * len(dists)), replace=False))
        trn = lab[(lab['T'] <= v) & ~lab.district_id.isin(pseudo)]
        val = lab[lab.O == v]
        vs, vu = val[~val.district_id.isin(pseudo)], val[val.district_id.isin(pseudo)]
        Ps, Pu = run_cfgs(trn, vs, vu, seeds=(SEED,))
        P = widen_finalize(np.r_[Ps, Pu]); y = np.r_[vs.brgi.values, vu.brgi.values]
        h = np.r_[vs.horizon.values, vu.horizon.values]
        print(V, 'weighted pinball =', round(score(y, P[:, 0], P[:, 1], P[:, 2], h), 4))

## 7. Final model: train on all labelled rows and write `solution.csv`

In [8]:
SEEDS = (42, 7, 2026, 11, 99)
lab = df[df._is_test == 0]
tst = df[df._is_test == 1]
is_seen = tst.district_id.isin(set(lab.district_id)).values
print('test rows:', len(tst), '| unseen-district rows:', (~is_seen).sum())
Ps, Pu = run_cfgs(lab, tst[is_seen], tst[~is_seen], seeds=SEEDS)
P = np.zeros((len(tst), 3)); P[is_seen] = Ps; P[~is_seen] = Pu
P = widen_finalize(P)
sub = pd.DataFrame({'id': tst.id.values, 'p10': P[:, 0], 'p50': P[:, 1], 'p90': P[:, 2]})
ss = pd.read_csv(DATA + 'sample_submission.csv')
sub = ss[['id']].merge(sub, on='id', how='left')
assert len(sub) == len(ss) and sub.notna().all().all()
assert np.isfinite(sub[['p10', 'p50', 'p90']].values).all()
assert (sub.p10 <= sub.p50).all() and (sub.p50 <= sub.p90).all()
sub.to_csv('solution.csv', index=False, float_format='%.4f')
sub.describe()

test rows: 2472 | unseen-district rows: 368


,p10,p50,p90
count,2472.000000,2472.000000,2472.000000
mean,35.924691,49.246640,62.527701
std,7.971302,8.816797,8.038903
min,7.691480,11.684022,31.016657
25%,30.690109,43.640448,57.464398
50%,36.019199,50.120456,63.549993
75%,41.680440,55.591814,68.218348
max,60.252879,75.390868,84.477682
